In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample.to_csv("submission.csv", index =False)

# **EDA** 

In [ ]:
import hashlib
import matplotlib.pyplot as plt
import re
from collections import Counter
import warnings
import random
import joblib
import gc
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

import lightgbm as lgb
import xgboost as xgb

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from scipy.sparse import csr_matrix, hstack, vstack
import wandb

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_options = 5

LABEL2IDX= { "A" :0, "B": 1, "C" : 2, "D" : 3, "E" : 4}

IDX2LABEL = {v:k for k,v in LABEL2IDX.items()}


DATA_DIR = "/kaggle/input/"

OUTPUT_DIR = "/kaggle/working/"

os.makedirs(OUTPUT_DIR, exist_ok = True)



In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
train.head()
test.head()

Shape and columns of train and test data

In [ ]:
print("Train shape:", train.shape)
print("\nTest shape:", test.shape)
print("\nTrain cloumns:", list(train.columns))
print("\nTest columns:", list(test.columns))

Missing values in train and test datasets

In [ ]:

def clean_text(text):

    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub("<.*?>"," ",text)
    text = re.sub("\s+"," ",text)
    return text.strip()

def apk(actual,predicted,k=3):

    predicted = predicted[:k]
    score = 0
    hits = 0
    for i,p in enumerate(predicted):
        if p==actual and p not in predicted[:i]:
            hits += 1
            score += hits/(i+1)
    return score

def mapk(actuals,predicteds,k=3):

    return np.mean([apk(a,p,k) for a,p in zip(actuals,predicteds)])
def prediction_to_labels(probabilities):
    top = np.argsort(-probabilities,axis=1)
    preds = [[IDX2LABEL[i] for i in row[:3]] for row in top]
    return preds
def evaluate(probabilities,answers):

    preds = prediction_to_labels(probabilities)

    map3 = mapk(answers,preds)

    top1 = np.argmax(probabilities,axis=1)

    acc = accuracy_score([LABEL2IDX[a] for a in answers],top1)

    f1 = f1_score([LABEL2IDX[a] for a in answers],top1,average="macro")

    print(f"Accuracy : {acc:.5f}")

    print(f"Macro F1 : {f1:.5f}")

    print(f"MAP@3     : {map3:.5f}")

    return acc,f1,map3
def strip_boilerplate(text):
    text = clean_text(text)
    patterns = [
        r"pick the best possible answer:?",
        r"choose the correct answer:?",
        r"which of the following is correct:?",
        r"which of the following statements is true:?",
        r"select the best option:?",
        r"what is the correct answer:?"
    ]

    for pattern in patterns:
        text = re.sub(pattern,"",text,flags=re.IGNORECASE)

    return text.strip()

In [ ]:
print("Missing values in train:", train.isnull().sum())
print("Missing values in test:", test.isnull().sum())

In [ ]:
answer_counts = train["answer"].value_counts()
print(answer_counts)

In [ ]:
answer_counts.sort_index().plot(kind="bar", color="b",title="Answers distribution in training data")
plt.ylabel("Count")

In [ ]:
dup_in_train = train.duplicated(subset=["prompt", "A","B","C","D","E"]).sum()
print("Total duplicates rows/questions which also have same options :",dup_in_train)

In [ ]:
OPTIONS = ["A","B","C","D","E"]
def option_hash(row):
    key = "|".join(str(row[c]).strip().lower() for c in OPTIONS)
    return hashlib.md5(key.encode()).hexdigest()

train["Question_hash"] = train.apply(option_hash, axis=1)
test["Question_hash"] = test.apply(option_hash, axis=1)
print("Train Hashes:\n",train["Question_hash"])
print("Test Hashes:\n",test["Question_hash"])

In [ ]:
print("Unique Questions in Train out of total questions:\n", train["Question_hash"].nunique(),"/",len(train))
print("Unique Questions in Test out of total questions:\n", test["Question_hash"].nunique(),"/",len(test))

In [ ]:
overlap_hash = set(test["Question_hash"]).intersection(set(train["Question_hash"]))
pct = len(overlap_hash) / test["Question_hash"].nunique() * 100
print(f"{len(overlap_hash)} of {test['Question_hash'].nunique()} unique test questions ")
print(f"({pct:.1f}%) reuse the exact same option set as a train question.")

In [ ]:
answers_per_group = train.groupby("Question_hash")["answer"].nunique()
n_inconsistent = (answers_per_group > 1).sum()
print("Question groups with inconsistent answers:",n_inconsistent)
print("out of",train['Question_hash'].nunique())

In [ ]:
def longest_option(row):
    lengths = {opt: len(str(row[opt])) for opt in OPTIONS}
    return max(lengths, key=lengths.get)
 
train["longest_opt"] = train.apply(longest_option, axis=1)
match_rate = (train["longest_opt"] == train["answer"]).mean()*100
print(f"Longest-option heuristic accuracy: {match_rate:.3f}")

In [ ]:
dedup = train.drop_duplicates(subset="Question_hash")
print(f"Answer count after removing the duplicates{dedup["answer"].value_counts()}")
print(f"Answer count before the removing the duplicates:{train["answer"].value_counts()}")

In [ ]:
for column in ["prompt"] + OPTIONS:
    n_empty = (train[column].astype(str).str.strip() == "").sum()
    print(f"Empty or whitespaces values in '{column}':{n_empty}")
def has_non_ascii(s):
    return any(ord(c) >127 for c in str(s))
n_non_ascii = train["prompt"].apply(has_non_ascii).sum()
print(f"Prompts containing non-ASCII characters: {n_non_ascii}")
if n_non_ascii:
    print(train.loc[train["prompt"].apply(has_non_ascii), "prompt"].head(3).tolist())

In [ ]:
def clean_prompt(p):
    p = re.sub(r"^[A-Za-z ,]+:\s*","",str(p))
    p = re.sub(r"(among the listed options\.|frm the following choices\.|carefully|.)$","",p)
    return p
    
def word_overlap(prompt_words, opt_text):
    opt_words = set(re.findall(r"\w+", str(opt_text).lower()))
    return len(prompt_words & opt_words)

ranks = []
for _, row in train.iterrows():
    prompt_words = set(re.findall(r"\w+", clean_prompt(row["prompt"]).lower()))
    overlaps = {opt : word_overlap(prompt_words, row[opt]) for opt in OPTIONS}
    sorted_opts = sorted(overlaps, key=lambda k: -overlaps[k])
    ranks.append(sorted_opts.index(row["answer"]) + 1)


train["overlap_rank"] = ranks
print(f"Correct answer's word overlap rank vs the prompt: {train["overlap_rank"].value_counts().sort_index()}")
print(f"Mean rank: {train["overlap_rank"].mean():.5f}")

In [ ]:
n_test_dupe_prompts = test["prompt"].duplicated().sum()
n_test_dupe_full = test.duplicated(subset=["prompt"] + OPTIONS).sum()
print(f"Duplicate prompts in test: {n_test_dupe_prompts}")
print(f"Fully duplicate rows in test: {n_test_dupe_full}")


domains = {
    "Physics": ["quantum", "energy", "force", "particle", "velocity", "gravity", "wave", "electron", "magnetic", "relativity"],
    "Biology": ["cell", "organism", "gene", "protein", "species", "evolution", "enzyme", "dna"],
    "Chemistry": ["molecule", "reaction", "acid", "compound", "element", "bond", "solution"],
    "Astronomy": ["star", "planet", "galaxy", "orbit", "universe", "solar", "black hole"],
    "Philosophy": ["ethics", "existential", "philosopher", "metaphysic", "moral", "consciousness"],
    "CS/Math": ["algorithm", "function", "equation", "matrix", "probability", "computation"],
}

def tag_domain(text):
    t = str(text).lower()
    for domain, kws in domains.items():
        if any(kw in t for kw in kws):
            return domain
    return "Other/Unclassified"
 
train["domain_guess"] = train["prompt"].apply(tag_domain)
print(train["domain_guess"].value_counts())

all_words = []
for p in train["prompt"]:
    cleaned = clean_prompt(p).lower()
    all_words.extend(re.findall(r"[a-z]{4,}", cleaned))  # words with 4+ letters
 
stopwords = {"what", "which", "does", "that", "with", "from", "this", "have",
             "following", "best", "most", "would", "when", "should"}
filtered = [w for w in all_words if w not in stopwords]
top_words = Counter(filtered).most_common(20)
print("Top 20 content words across all prompts:")
for word, count in top_words:
    print(f"  {word}: {count}")


In [ ]:
train["prompt_len"] = train["prompt"].str.len()

plt.figure(figsize=(8,4))
train["prompt_len"].hist(bins=50)
plt.title("Prompt Length Distribution")
plt.show()

In [ ]:
for col in OPTIONS:
    train[col+"_len"] = train[col].astype(str).str.len()
length_df = train[[c+"_len" for c in OPTIONS]]

length_df.boxplot(figsize=(8,4))

# **Feature Enginerring**


In [ ]:
train["option_set"] = train[["A","B","C","D","E"]].apply(lambda x: " | ".join([str(i).strip() for i in x]),axis=1)

test["option_set"] = test[["A","B","C","D","E"]].apply(lambda x: " | ".join([str(i).strip() for i in x]),axis=1)

lookup_map = {}

for _,row in train.iterrows():

    lookup_map[row["option_set"]] = row["answer"]

print(f"Lookup size : {len(lookup_map)}")

In [ ]:
all_prompt = pd.concat([train["prompt"],test["prompt"]]).apply(clean_text)
all_options =[]
for col in ["A","B","C","D","E"]:
    all_options.extend(train[col].apply(clean_text))
    all_options.extend(test[col].apply(clean_text))

corpus = list(all_prompt) + all_options

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2),stop_words="english",sublinear_tf=True)

tfidf.fit(corpus)

print("vocabulary:", len(tfidf.vocabulary_))


In [ ]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

def build_joint_text(df):

    pairs = []

    for _, row in df.iterrows():

        prompt = strip_boilerplate(
            clean_text(str(row["prompt"]))
        )

        for col in OPTION_COLS:

            option = clean_text(str(row[col]))

            pairs.append(prompt + " " + option)

    return pairs


train_joint = build_joint_text(train)
test_joint = build_joint_text(test)

joint_tfidf = TfidfVectorizer(
    max_features=4000,
    ngram_range=(1,2),
    stop_words="english",
    sublinear_tf=True
)

joint_tfidf.fit(train_joint)

X_joint_train = joint_tfidf.transform(train_joint)
X_joint_test = joint_tfidf.transform(test_joint)

print("Joint TF-IDF ready.")

In [ ]:
svd = TruncatedSVD(n_components=16, random_state=42)

svd.fit(tfidf.transform(corpus))


In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(tfidf, "models/tfidf.pkl")
joblib.dump(joint_tfidf, "models/joint_tfidf.pkl")
joblib.dump(svd, "models/svd.pkl")

print("Vectorizers saved successfully.")

In [ ]:
NEGATION_WORDS = {
    "no","not","never","none","neither","nor",
    "cannot","without","n't","false","unlike"
}
ABSOLUTE_WORDS = {
    "always","all","every","entirely",
    "absolutely","completely","only",
    "must","true"
}
UNITS_WORDS = {
    "kg","cm","mm","m","km","hz",
    "mol","nm","v","w","ev","kj",
    "pa","atm","sec","min","hr"
}
def get_longest_common_prefix(strings):

    if len(strings)==0:
        return ""

    s1=min(strings)
    s2=max(strings)

    for i,c in enumerate(s1):

        if c!=s2[i]:
            return s1[:i]
    return s1
def jaccard_similarity(a,b):
    a=set(a)
    b=set(b)
    return len(a&b)/(len(a|b)+1e-9)
def cosine_score(vec1,vec2):

    return cosine_similarity(vec1,vec2)[0,0]
def similarity_features(prompt_words,
                        option_words,
                        prompt_vec,
                        option_vec):

    overlap=len(

        prompt_words.intersection(option_words)
    )
    union=len(
        prompt_words.union(option_words)
    )
    jaccard=overlap/(union+1e-6)
    cosine=cosine_score(
        prompt_vec,
        option_vec
    )
    overlap_ratio=overlap/(
        min(len(prompt_words),
            len(option_words))+1e-6
    )
    return {
        "cosine":cosine,
        "jaccard":jaccard,
        "overlap":overlap_ratio}

In [ ]:
def length_features(option_length,all_lengths,rank):
    mean=np.mean(all_lengths)
    std=np.std(all_lengths)+1e-6
    return {
        "len_rank":rank,
        "len_z":(option_length-mean)/std,
        "len_ratio":option_length/(mean+1e-6),
        "longest":int(rank==1),
        "shortest":int(rank==5)
    }

def lookup_features(option_set,option_label):

    if option_set in lookup_map:
        return {"known":1,"lookup_match":int(lookup_map[option_set]==option_label)}

    return {
        "known":0,
        "lookup_match":0
    }

In [ ]:
def option_statistics(text):

    words=text.lower().split()

    chars=len(text)

    alpha=max(

        sum(c.isalpha() for c in text),

        1

    )

    return {

        "negation":

        sum(

            w in NEGATION_WORDS

            for w in words

        ),

        "absolute":
        sum(
            w in ABSOLUTE_WORDS
            for w in words
        ),
        "digit_ratio":
        sum(
            c.isdigit()
            for c in text
        )/(chars+1e-6),
        "uppercase_ratio":
        sum(c.isupper() for c in text)/alpha,"contains_unit": int(any(w in UNITS_WORDS for w in words))}

In [ ]:
OPTION_COLS = ["A", "B", "C", "D", "E"]

def prepare_text_data(df):
    """
    Precompute all text representations required for feature engineering.
    """

    data = {}

    data["prompt_clean"] = df["prompt"].fillna("").apply(clean_text)

    data["prompt_core"] = (
        df["prompt"]
        .fillna("")
        .apply(strip_boilerplate)
    )


    data["prompt_tfidf"] = tfidf.transform(
        data["prompt_core"]
    )

    data["prompt_svd"] = svd.transform(
        data["prompt_tfidf"]
    )

    data["options_clean"] = {}

    data["options_tfidf"] = {}

    data["options_svd"] = {}

    for col in OPTION_COLS:

        texts = (
            df[col]
            .fillna("")
            .astype(str)
            .apply(clean_text)
        )

        data["options_clean"][col] = texts

        tfidf_vec = tfidf.transform(texts)

        data["options_tfidf"][col] = tfidf_vec

        data["options_svd"][col] = svd.transform(
            tfidf_vec
        )

    data["option_set"] = df[OPTION_COLS].apply(

        lambda x: " | ".join(
            str(v).strip() for v in x
        ),

        axis=1

    ).values

    return data


print("Feature preparation utilities ready.")

In [ ]:
def basic_features(
    row_idx,
    option_idx,
    option_label,
    text_data,
    lookup_map
):
    """
    Returns a dictionary of basic engineered features.
    """

    feat = {}

    prompt_text = text_data["prompt_core"].iloc[row_idx]

    prompt_words = prompt_text.lower().split()

    prompt_word_cnt = max(len(prompt_words), 1)

    prompt_char_cnt = max(len(prompt_text), 1)

    option_text = (
        text_data["options_clean"][option_label]
        .iloc[row_idx]
    )

    option_words = option_text.lower().split()

    option_word_cnt = max(len(option_words), 1)

    option_char_cnt = len(option_text)

    row_options = [

        text_data["options_clean"][c].iloc[row_idx]

        for c in OPTION_COLS

    ]

    char_lengths = [

        len(x)

        for x in row_options

    ]

    word_lengths = [

        len(x.split())

        for x in row_options

    ]

    mean_char = np.mean(char_lengths)

    std_char = np.std(char_lengths) + 1e-6

    mean_word = np.mean(word_lengths)

    std_word = np.std(word_lengths) + 1e-6

    ranks = (

        pd.Series(char_lengths)

        .rank(
            ascending=False,
            method="min"
        )

        .values

    )

    option_set = text_data["option_set"][row_idx]

    feat["lookup_seen"] = int(
        option_set in lookup_map
    )

    feat["lookup_match"] = int(

        lookup_map.get(option_set, None)

        == option_label

    )

    feat["char_len"] = option_char_cnt

    feat["word_len"] = option_word_cnt

    feat["len_rank"] = ranks[option_idx]

    feat["is_longest"] = int(
        ranks[option_idx] == 1
    )

    feat["is_shortest"] = int(
        ranks[option_idx] == 5
    )

    feat["char_zscore"] = (

        option_char_cnt - mean_char

    ) / std_char

    feat["word_zscore"] = (

        option_word_cnt - mean_word

    ) / std_word

    feat["char_ratio"] = (

        option_char_cnt

        / (mean_char + 1e-6)

    )

    feat["word_ratio"] = (

        option_word_cnt

        / (mean_word + 1e-6)

    )

    feat["prompt_char_ratio"] = (

        option_char_cnt

        / prompt_char_cnt

    )

    feat["prompt_word_ratio"] = (

        option_word_cnt

        / prompt_word_cnt

    )

    feat["lcp_length"] = len(

        get_longest_common_prefix(

            row_options

        )

    )

    return feat

In [ ]:
def similarity_features(
    row_idx,
    option_idx,
    option_label,
    text_data
):
    """
    Semantic similarity features between prompt and option.
    """

    feat = {}

    prompt_text = text_data["prompt_core"].iloc[row_idx]

    prompt_words = set(
        prompt_text.lower().split()
    )

    prompt_vec = (
        text_data["prompt_tfidf"][row_idx]
    )

    prompt_svd = (
        text_data["prompt_svd"][row_idx]
    )

    option_text = (
        text_data["options_clean"][option_label]
        .iloc[row_idx]
    )

    option_words = set(
        option_text.lower().split()
    )

    option_vec = (
        text_data["options_tfidf"][option_label][row_idx]
    )

    option_svd = (
        text_data["options_svd"][option_label][row_idx]
    )

    intersection = len(
        prompt_words & option_words
    )

    union = len(
        prompt_words | option_words
    )

    feat["word_overlap"] = intersection

    feat["overlap_ratio"] = (

        intersection /

        (min(len(prompt_words),
             len(option_words)) + 1e-6)

    )

    feat["jaccard"] = (

        intersection /

        (union + 1e-6)

    )

    feat["tfidf_cosine"] = cosine_similarity(
        prompt_vec,
        option_vec
    )[0, 0]

    feat["svd_cosine"] = (

        np.dot(prompt_svd, option_svd)

        /

        (

            np.linalg.norm(prompt_svd)

            *

            np.linalg.norm(option_svd)

            + 1e-6

        )

    )

    from scipy.sparse import vstack
    
    option_vectors = vstack([
        text_data["options_tfidf"][c][row_idx]
        for c in OPTION_COLS
    ])
    
    sim_matrix = cosine_similarity(option_vectors)
    
    sims = np.delete(sim_matrix[option_idx], option_idx)
    
    feat["opt_sim_mean"] = np.mean(sims)
    feat["opt_sim_max"] = np.max(sims)
    feat["opt_sim_min"] = np.min(sims)
    feat["opt_sim_std"] = np.std(sims)

    row_options = [

        text_data["options_clean"][c]
        .iloc[row_idx]

        for c in OPTION_COLS

    ]

    prefix = get_longest_common_prefix(
        row_options
    )

    feat["common_prefix_len"] = len(prefix)

    feat["option_prefix_ratio"] = (

        len(prefix)

        /

        (len(option_text) + 1e-6)

    )

    return feat

In [ ]:


def advanced_features(
    row_idx,
    option_idx,
    option_label,
    text_data
):
    """
    Advanced handcrafted linguistic features.
    """

    feat = {}

    option_text = (
        text_data["options_clean"][option_label]
        .iloc[row_idx]
    )

    words = option_text.lower().split()

    word_count = max(len(words), 1)

    char_count = max(len(option_text), 1)

    neg_count = sum(
        w in NEGATION_WORDS
        for w in words
    )

    feat["negation_count"] = neg_count

    feat["negation_ratio"] = (
        neg_count / word_count
    )

    abs_count = sum(
        w in ABSOLUTE_WORDS
        for w in words
    )

    feat["absolute_count"] = abs_count

    feat["absolute_ratio"] = (
        abs_count / word_count
    )

    stop_count = sum(
        w in STOPWORDS
        for w in words
    )

    feat["stopword_ratio"] = (
        stop_count / word_count
    )

    digit_count = sum(
        c.isdigit()
        for c in option_text
    )

    feat["digit_ratio"] = (
        digit_count / char_count
    )

    feat["contains_digit"] = int(
        digit_count > 0
    )


    unit_present = any(
        w in UNITS_WORDS
        for w in words
    )

    feat["contains_unit"] = int(
        unit_present
    )


    alpha_count = sum(
        c.isalpha()
        for c in option_text
    )

    upper_count = sum(
        c.isupper()
        for c in option_text
    )

    feat["uppercase_ratio"] = (
        upper_count /
        (alpha_count + 1e-6)
    )

    feat["question_marks"] = option_text.count("?")

    feat["comma_count"] = option_text.count(",")

    feat["semicolon_count"] = option_text.count(";")

    feat["colon_count"] = option_text.count(":")

    feat["parenthesis_count"] = (
        option_text.count("(")
        + option_text.count(")")
    )

    unique_words = len(set(words))

    feat["lexical_diversity"] = (
        unique_words / word_count
    )

    feat["avg_word_length"] = (
        np.mean([len(w) for w in words])
        if words else 0
    )

    svd_vec = (
        text_data["options_svd"][option_label][row_idx]
    )

    feat["svd_mean"] = np.mean(svd_vec)

    feat["svd_std"] = np.std(svd_vec)

    feat["svd_max"] = np.max(svd_vec)

    feat["svd_min"] = np.min(svd_vec)

    feat["svd_sum"] = np.sum(svd_vec)

    feat["svd_norm"] = np.linalg.norm(svd_vec)

    return feat

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)
def extract_features(
    df,
    text_data,
    lookup_map
):
    """
    Generate engineered features for all
    (question, option) pairs.
    """

    feature_rows = []

    labels = []

    groups = []

    for row_idx in tqdm(range(len(df)), desc="Extracting Features"):

        for option_idx, option_label in enumerate(OPTION_COLS):

            feat = {}

            feat.update(

                basic_features(
                    row_idx,
                    option_idx,
                    option_label,
                    text_data,
                    lookup_map
                )

            )

            feat.update(

                similarity_features(
                    row_idx,
                    option_idx,
                    option_label,
                    text_data
                )

            )

            feat.update(

                advanced_features(
                    row_idx,
                    option_idx,
                    option_label,
                    text_data
                )

            )

            feature_rows.append(feat)

            if "answer" in df.columns:

                labels.append(

                    int(
                        option_label
                        == df.iloc[row_idx]["answer"]
                    )

                )

            groups.append(row_idx)

    feature_df = pd.DataFrame(feature_rows)

    return feature_df, labels, groups


train_text = prepare_text_data(train)

X_train_df, y, groups = extract_features(
    train,
    train_text,
    lookup_map
)

print(X_train_df.shape)


test_text = prepare_text_data(test)

X_test_df, _, _ = extract_features(
    test,
    test_text,
    lookup_map
)

print(X_test_df.shape)

def build_joint_text(df):

    texts = []

    for _, row in df.iterrows():

        prompt = strip_boilerplate(
            clean_text(row["prompt"])
        )

        for col in OPTION_COLS:

            texts.append(
                prompt + " " + clean_text(str(row[col]))
            )

    return texts


train_joint = build_joint_text(train)

test_joint = build_joint_text(test)

X_joint_train = joint_tfidf.transform(train_joint)

X_joint_test = joint_tfidf.transform(test_joint)

X_train_dense = csr_matrix(
    X_train_df.values
)

X_test_dense = csr_matrix(
    X_test_df.values
)

X_train = hstack([
    X_train_dense,
    X_joint_train
]).tocsr()

X_test = hstack([
    X_test_dense,
    X_joint_test
]).tocsr()

dense_feature_names = X_train_df.columns.tolist()

feature_names = (
    dense_feature_names +
    [f"tfidf_{i}" for i in range(X_joint_train.shape[1])]
)
print("Train:", X_train.shape)
print("Test :", X_test.shape)

In [ ]:
LABELS = ["A", "B", "C", "D", "E"]

label_to_index = {
    label: idx
    for idx, label in enumerate(LABELS)
}

index_to_label = {
    idx: label
    for idx, label in enumerate(LABELS)
}

In [ ]:
print("STOPWORDS" in globals())
print("NEGATION_WORDS" in globals())
print("ABSOLUTE_WORDS" in globals())
print("UNITS_WORDS" in globals())
print("OPTION_COLS" in globals())
print("label_to_index" in globals())
print("index_to_label" in globals())

In [ ]:
train_text = prepare_text_data(train)
test_text = prepare_text_data(test)

X_train_df, y, groups = extract_features(
    train,
    train_text,
    lookup_map
)

X_test_df, _, _ = extract_features(
    test,
    test_text,
    lookup_map
)

print(X_train_df.shape)
print(X_test_df.shape)

In [ ]:
from scipy.sparse import csr_matrix, hstack

X_train = hstack([
    csr_matrix(X_train_df.values),
    X_joint_train
]).tocsr()

X_test = hstack([
    csr_matrix(X_test_df.values),
    X_joint_test
]).tocsr()

print(X_train.shape)
print(X_test.shape)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(len(y))
print(len(groups))
print(np.isnan(X_train_df.values).sum())
print(np.isnan(X_test_df.values).sum())
print(X_train_dense.shape)
print(X_joint_train.shape)
print(X_train.shape)

In [ ]:
y = np.array(y, dtype=np.int8)
groups = np.array(groups)
print(type(y))
print(y[:10])
print(np.unique(y, return_counts=True))

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss
import joblib


N_FOLDS = 5

gkf = GroupKFold(n_splits=N_FOLDS)

lgb_params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "n_estimators": 5000,
    "num_leaves": 63,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity":-1
}
callbacks=[
    early_stopping(200),
    log_evaluation(100),
]


oof_lgb = np.zeros(len(y))
test_lgb = np.zeros(X_test.shape[0])

lgb_models = []

for fold, (train_idx, valid_idx) in enumerate(
    gkf.split(X_train, y, groups),
    start=1,
):

    print("=" * 60)
    print(f"Fold {fold}")

    X_tr = X_train[train_idx]
    X_va = X_train[valid_idx]

    y_tr = y[train_idx]
    y_va = y[valid_idx]

    model = LGBMClassifier(**lgb_params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="binary_logloss",
        callbacks=[
            early_stopping(200),
            log_evaluation(100),
        ],
    )

    oof_lgb[valid_idx] = model.predict_proba(X_va)[:, 1]

    test_lgb += (
        model.predict_proba(X_test)[:, 1] / N_FOLDS
    )

    lgb_models.append(model)

    print(
        "Fold LogLoss:",
        log_loss(y_va, oof_lgb[valid_idx]),
    )

print("Overall LogLoss")

overall = log_loss(y, oof_lgb)

print(overall)

joblib.dump(lgb_models, "models/lightgbm_models.pkl")

In [ ]:
print("Engineered features :", len(feature_names))
print("TF-IDF features     :", X_joint_train.shape[1])
print("Total expected      :", len(feature_names) + X_joint_train.shape[1])
print("Model features      :", len(lgb_models[0].feature_importances_))
print(X_train.shape)

In [ ]:
print(feature_names[:10])
print(feature_names[45:55])
import pandas as pd

imp = pd.DataFrame({
    "feature": feature_names,
    "importance": lgb_models[0].feature_importances_
})

imp = imp.sort_values("importance", ascending=False)

print(imp.head(50))

In [ ]:
for var in [
    "train_feat",
    "X_train_dense",
    "X_joint_train",
    "X_train",
    "feature_cols",
    "dense_feature_names",
    "lgb_models"
]:
    print(var, "->", var in globals())

In [ ]:
print(X_train_dense)
print(X_train_dense.shape)

# NEXT model

In [ ]:
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupKFold
import numpy as np

gkf = GroupKFold(n_splits=5)
oof_cat = np.zeros(len(y))
test_pred_cat = np.zeros(X_test.shape[0])

cat_models = []

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X_train, y, groups)):

    print("=" * 50)
    print(f"Fold {fold+1}")

    train_pool = Pool(
        X_train[train_idx],
        y[train_idx]
    )

    valid_pool = Pool(
        X_train[valid_idx],
        y[valid_idx]
    )

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=200,
        task_type="GPU",
        devices="0"
    )
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=200
    )

    oof_cat[valid_idx] = model.predict_proba(
        X_train[valid_idx]
    )[:, 1]

    test_pred_cat += (
        model.predict_proba(X_test)[:, 1]
        / gkf.n_splits
    )

    cat_models.append(model)

    print(
        "Fold LogLoss:",
        log_loss(
            y[valid_idx],
            oof_cat[valid_idx]
        )
    )

print("Overall OOF:", log_loss(y, oof_cat))

# **Milestone 1** #

In [ ]:
#question 1 
counts = train['answer'].value_counts()

print("Frequency Distribution:")
print(counts)

most_frequent = counts.max()
least_frequent = counts.min()

total_sum = most_frequent + least_frequent

print(f"\nMost frequent count: {most_frequent}")
print(f"Least frequent count: {least_frequent}")
print(f"Sum of most and least frequent options: {total_sum}")

In [ ]:
#question 2 
import string
unique_words = set()
remove_punct_map = str.maketrans('', '', string.punctuation)
for text in train['prompt']:
    if isinstance(text, str): # Ensure the value is a string
        cleaned_text = text.lower().translate(remove_punct_map)
        
        unique_words.update(cleaned_text.split())
vocab_size = len(unique_words)

print(f"Total number of unique words (vocabulary size): {vocab_size}")

In [ ]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

text = train['prompt'].iloc[0]

if isinstance(text, str):
    remove_punct_map = str.maketrans('', '', string.punctuation)
    
    cleaned_words = text.lower().translate(remove_punct_map).split()
    
    filtered_words = [word for word in cleaned_words if word not in ENGLISH_STOP_WORDS]
    
    words_left = len(filtered_words)
    
    print(f"Original word count: {len(cleaned_words)}")
    print(f"Words left after filtering: {words_left}")
else:
    print("The prompt in this row is not a string.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cols_to_combine = ['prompt', 'A', 'B', 'C', 'D', 'E']

actual_cols = [col for col in cols_to_combine if col in train.columns]

combined_text_series = train[actual_cols].fillna('').astype(str).agg(' '.join, axis=1)
combined_text_list = combined_text_series.tolist()

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text_list)

vocab_size = X.shape[1]

print(f"Total number of feature columns (vocabulary size): {vocab_size}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_text = str(train['prompt'].iloc[0])
option_a_text = str(train['A'].iloc[0])

prompt_vector = vectorizer.transform([prompt_text])
option_a_vector = vectorizer.transform([option_a_text])

similarity_matrix = cosine_similarity(prompt_vector, option_a_vector)

similarity_score = round(similarity_matrix[0][0], 5)

print(f"Cosine similarity between prompt and option A (Row 1): {similarity_score}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

correct_predictions = 0
total_rows = len(train)
options = ['A', 'B', 'C', 'D', 'E']

for index, row in train.iterrows():
    prompt_text = str(row['prompt'])
    prompt_vec = vectorizer.transform([prompt_text])
    
    similarities = {}
    
    for opt in options:
        opt_text = str(row[opt])
        opt_vec = vectorizer.transform([opt_text])
        
        sim_score = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities[opt] = sim_score
    
    best_option = max(similarities, key=similarities.get)
    
    if best_option == row['answer']:
        correct_predictions += 1

accuracy_percentage = (correct_predictions / total_rows) * 100

print(f"Total rows evaluated: {total_rows}")
print(f"Correct predictions: {correct_predictions}")
print(f"Accuracy of TF-IDF Cosine Similarity approach: {accuracy_percentage:.2f}%")

In [ ]:
def calculate_ap3(ground_truth, predictions):
    """
    Calculates the Average Precision at 3 (AP@3) for a single question.
    """
    score = 0.0
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            score = 1.0 / (i + 1)
            break 
            
    return score

actual_answer = 'C'
model_predictions = ['C', 'A', 'B']

map3_score = calculate_ap3(actual_answer, model_predictions)

print(f"Ground Truth: {actual_answer}")
print(f"Predictions: {model_predictions}")
print(f"MAP@3 Score: {map3_score}")

In [ ]:
counts = train['answer'].value_counts()
top_3_answers = counts.nlargest(3).index.tolist()

print(f"Top 3 most frequent answers (Static Prediction): {top_3_answers}")

def calculate_ap3(ground_truth, predictions):
    score = 0.0
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            score = 1.0 / (i + 1)
            break
    return score

total_map3_score = 0.0
total_rows = len(train)

for index, row in train.iterrows():
    actual_answer = row['answer']
    
    total_map3_score += calculate_ap3(actual_answer, top_3_answers)

final_map3_score = total_map3_score / total_rows

print(f"Overall MAP@3 score of Majority Class baseline: {final_map3_score:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

options = ['A', 'B', 'C', 'D', 'E']

def calculate_ap3(ground_truth, predictions):
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            return 1.0 / (i + 1)
    return 0.0

total_map3_score = 0.0
total_rows = len(train)

for index, row in train.iterrows():
    prompt_text = str(row['prompt'])
    
    prompt_vec = vectorizer.transform([prompt_text])
    
    similarity_scores = {}
    for opt in options:
        opt_text = str(row[opt])
        opt_vec = vectorizer.transform([opt_text])
        
        sim_score = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarity_scores[opt] = sim_score

    sorted_options = sorted(similarity_scores, key=similarity_scores.get, reverse=True)
    
    top_3_predictions = sorted_options[:3]
    
    actual_answer = row['answer'] 
    total_map3_score += calculate_ap3(actual_answer, top_3_predictions)

final_map3_score = total_map3_score / total_rows

print(f"Pipeline Evaluation Summary")
print(f"---------------------------")
print(f"Total Rows Processed: {total_rows}")
print(f"Final Average MAP@3 Score: {final_map3_score:.4f}")